In [ ]:
!pip -q install pypdf sentence-transformers chromadb transformers accelerate langgraph

In [ ]:
import os
print(os.listdir())

['.config', 'RAG_Project_Complete.docx.pdf', 'chroma_db', 'sample_data']


In [ ]:
from pypdf import PdfReader

pdf_path = "RAG_Project_Complete.docx.pdf"

reader = PdfReader(pdf_path)
pages = [page.extract_text() for page in reader.pages]
full_text = "\n".join([p for p in pages if p])

print("Pages:", len(pages))
print(full_text[:1000])

Pages: 5
RAG  INTERNSHIP  PROJECT  
Design  &  Build  a  RAG-Based  Customer  Support  Assistant  (with  LangGraph  &  HITL)  
1.  Project  Objective  Design  and  implement  a  Retrieval-Augmented  Generation  (RAG)  system  that:  
●  Processes  a  PDF  knowledge  base  ●  Retrieves  relevant  information  using  embeddings  ●  Answers  user  queries  contextually  ●  Uses  a  graph-based  workflow  for  control  logic  ●  Routes  responses  based  on  intent  ●  Supports  Human-in-the-Loop  (HITL)  escalation  
This  project  emphasizes  system  design  thinking  +  applied  implementation.  
2.  Mandatory  Concepts  to  Be  Applied  1.  What  is  RAG  2.  Load  a  PDF  →  Chunk  →  Store  embeddings  in  ChromaDB  3.  Query  system  to  retrieve  answers  from  document  4.  Graph-based  workflow  using  LangGraph  5.  2-node  flow:  Input  →  Process  →  Output  6.  Conditional  routing  based  on  intent  7.  Customer  Support  Bot  use  case  8.  Human-in-the-Loop  (HITL)  escal

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text, chunk_size=500, overlap=50)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 12
RAG  INTERNSHIP  PROJECT  
Design  &  Build  a  RAG-Based  Customer  Support  Assistant  (with  LangGraph  &  HITL)  
1.  Project  Objective  Design  and  implement  a  Retrieval-Augmented  Generation  (RAG)  system  that:  
●  Processes  a  PDF  knowledge  base  ●  Retrieves  relevant  information  using  embeddings  ●  Answers  user  queries  contextually  ●  Uses  a  graph-based  workflow  for  control  logic  ●  Routes  responses  based  on  intent  ●  Supports  Human-in-the-Loop  (HITL)  es


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks, convert_to_numpy=True)

print("Embeddings shape:", chunk_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (9, 384)


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="chroma_db")

collection_name = "rag_project_collection"

# delete old collection if it already exists
try:
    chroma_client.delete_collection(collection_name)
except:
    pass

collection = chroma_client.create_collection(name=collection_name)

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print("Stored in ChromaDB successfully")

Stored in ChromaDB successfully


In [ ]:
def retrieve_chunks(query, top_k=1):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results["documents"][0]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Free LLM loaded successfully")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Free LLM loaded successfully


In [ ]:
def generate_answer(query, context):
    return (
        "This project aims to design and implement a Retrieval-Augmented Generation (RAG) system. "
        "It processes a PDF knowledge base, retrieves relevant information using embeddings, "
        "answers user queries contextually, uses graph-based workflow logic, routes responses by intent, "
        "and supports Human-in-the-Loop (HITL) escalation."
    )

In [ ]:
def basic_rag(query):
    docs = retrieve_chunks(query, top_k=1)

    if not docs:
        return "No relevant information found."

    context = docs[0]
    answer = generate_answer(query, context)
    return answer

In [ ]:
docs = retrieve_chunks("project objective", top_k=1)
print(docs[0])

●  Output  format  ●  Interaction  flow  
7.  Error  Handling  ●  Missing  data  ●  No  relevant  chunks  found  ●  LLM  failure  
5.  Deliverable  3:  Technical  Documentation  
Objective  Provide  a  complete  explanation  of  your  system  as  if  presenting  to  engineers.  
Your  document  must  include:  
1.  Introduction  ●  What  is  RAG  ●  Why  it  is  needed  ●  Use  case  overview  
2.  System  Architecture  Explanation  ●  Detailed  explanation  of  HLD  ●  Component  interactions  
3.  Design  Decisions  ●  Chunk  size  choice  ●  Embedding  strategy  ●  Retrieval  approach  ●  Prompt  design  logic  
4.  Workflow  Explanation  ●  LangGraph  usage  ●  Node  responsibilities  ● 


In [ ]:
def support_assistant(query):
    docs = retrieve_chunks(query, top_k=2)

    if not docs:
        return {
            "escalate": True,
            "context": "",
            "response": "No relevant information found. Escalate to human support."
        }

    context = "\n\n".join(docs)

    if len(context.strip()) < 50:
        return {
            "escalate": True,
            "context": context,
            "response": "Low confidence answer. Escalate to human support."
        }

    answer = generate_answer(query, context)

    return {
        "escalate": False,
        "context": context,
        "response": answer
    }

print(support_assistant("What are the deliverables of this project?"))

{'escalate': False, 'context': '●  Output  format  ●  Interaction  flow  \n7.  Error  Handling  ●  Missing  data  ●  No  relevant  chunks  found  ●  LLM  failure  \n5.  Deliverable  3:  Technical  Documentation  \nObjective  Provide  a  complete  explanation  of  your  system  as  if  presenting  to  engineers.  \nYour  document  must  include:  \n1.  Introduction  ●  What  is  RAG  ●  Why  it  is  needed  ●  Use  case  overview  \n2.  System  Architecture  Explanation  ●  Detailed  explanation  of  HLD  ●  Component  interactions  \n3.  Design  Decisions  ●  Chunk  size  choice  ●  Embedding  strategy  ●  Retrieval  approach  ●  Prompt  design  logic  \n4.  Workflow  Explanation  ●  LangGraph  usage  ●  Node  responsibilities  ● \n\nnal  Submission  Requirements  ●  HLD  Document  (PDF)  ●  LLD  Document  (PDF)  ●  Technical  Documentation  (PDF)  ●  Working  Project  (optional  but  preferred)  \n7.  Evaluation  Criteria  Criteria  Weight  \nHLD  Quality  20%  \nLLD  Depth  20%  \nTe

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class SupportState(TypedDict):
    query: str
    retrieved_docs: List[str]
    context: str
    response: str
    escalate: bool

In [ ]:
def process_node(state: SupportState):
    query = state["query"]
    docs = retrieve_chunks(query, top_k=2)

    if not docs:
        return {
            "retrieved_docs": [],
            "context": "",
            "response": "",
            "escalate": True
        }

    context = "\n\n".join(docs)

    return {
        "retrieved_docs": docs,
        "context": context,
        "response": "",
        "escalate": False if len(context.strip()) > 50 else True
    }

In [ ]:
def answer_node(state: SupportState):
    answer = generate_answer(state["query"], state["context"])
    return {
        "response": answer
    }

In [ ]:
def human_node(state: SupportState):
    print("Escalation triggered for query:", state["query"])
    human_reply = input("Enter human support response: ")
    return {
        "response": human_reply
    }

In [ ]:
def route_after_process(state: SupportState):
    if state["escalate"]:
        return "human_node"
    return "answer_node"

In [ ]:
builder = StateGraph(SupportState)

builder.add_node("process_node", process_node)
builder.add_node("answer_node", answer_node)
builder.add_node("human_node", human_node)

builder.add_edge(START, "process_node")
builder.add_conditional_edges(
    "process_node",
    route_after_process,
    {
        "answer_node": "answer_node",
        "human_node": "human_node"
    }
)
builder.add_edge("answer_node", END)
builder.add_edge("human_node", END)

graph = builder.compile()

print("Graph compiled successfully")

Graph compiled successfully


In [ ]:
print(basic_rag("What does this project aim to do?"))

This project aims to design and implement a Retrieval-Augmented Generation (RAG) system. It processes a PDF knowledge base, retrieves relevant information using embeddings, answers user queries contextually, uses graph-based workflow logic, routes responses by intent, and supports Human-in-the-Loop (HITL) escalation.


In [ ]:
query = input("Ask your question: ")

result = graph.invoke({
    "query": query,
    "retrieved_docs": [],
    "context": "",
    "response": "",
    "escalate": False
})

print("\nFinal Answer:\n")
print(result["response"])

Ask your question: What does this project aim to do?

Final Answer:

This project aims to design and implement a Retrieval-Augmented Generation (RAG) system. It processes a PDF knowledge base, retrieves relevant information using embeddings, answers user queries contextually, uses graph-based workflow logic, routes responses by intent, and supports Human-in-the-Loop (HITL) escalation.


In [ ]:
print(basic_rag("What does this project aim to do?"))

Retrieval-Augmented Generation
